# 台股第二階段：時機評估與AI研究

WHID負責選股與估值；本版評估短中長期趨勢、外資投信、籌碼K線、市場／個股／新聞情緒、融資壓力與交易風險。新增情緒、融資壓力、籌碼K線及AI先作研究參考，不自動下單、不修改WHID分數或既有交易建議。詳細操作與限制請讀 README_第二階段.md。

In [ ]:
from pathlib import Path
import sys, importlib.util

# 可攜式定位：可從專案根目錄或 Stock_price_prediction 目錄開啟。
# 如果另存到其他專案，會使用該專案的相對位置，不綁定電腦上的絕對路徑。
START = Path.cwd().resolve()
search = [START, START / "Stock_price_prediction"]
for parent in START.parents:
    search.extend([parent, parent / "Stock_price_prediction"])
BASE = next((p for p in search if (p / "timing_tw.py").is_file() and (p / "config").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("找不到 Stock_price_prediction 程式目錄；請從專案根目錄或該資料夾開啟 Notebook")
BASE = BASE.resolve()
PROJECT_ROOT = BASE.parent

# 更新後即使未重啟Kernel，也不沿用先前從其他資料夾載入的同名模組。
module_prefixes = ("timing_us",) if "TW" == "US" else ("timing_data", "timing_rules", "timing_ai", "timing_tw")
for module_name in list(sys.modules):
    if any(module_name == p or module_name.startswith(p + ".") or module_name.startswith(p + "_") for p in module_prefixes):
        del sys.modules[module_name]
sys.path[:] = [p for p in sys.path if not (p and Path(p).name == "Stock_price_prediction")]
sys.path.insert(0, str(BASE.resolve()))

needed = {"numpy":"numpy", "pandas":"pandas", "yaml":"PyYAML", "yfinance":"yfinance",
          "openpyxl":"openpyxl", "sklearn":"scikit-learn", "xgboost":"xgboost", "torch":"torch"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
print("缺少套件：", missing or "無")
print("本次專案：", PROJECT_ROOT.resolve())
print("程式目錄：", BASE.resolve())
print("套件清單：", BASE / "requirements-timing.txt")


In [ ]:
# 僅在上一格顯示缺套件時才改為 True；安裝後重啟Kernel。
INSTALL_MISSING = False
if INSTALL_MISSING:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(BASE / "requirements-timing.txt")])
    print("安裝完成，請重新啟動Kernel並從第一格執行。")


In [ ]:
import pandas as pd
import timing_data, timing_tw
from timing_data import VERSION, load_config, candidates
from timing_tw import run, finlab_research_positions
for module in [timing_data, timing_tw]:
    if Path(module.__file__).resolve().parent != BASE.resolve():
        raise RuntimeError(f"模組路徑不一致：{module.__file__}")
print("程式版本：", VERSION)
print("資料模組：", Path(timing_data.__file__).resolve())
CONFIG_PATH = BASE / "config" / "timing_TW.yaml"
TICKERS = []  # 例如 ["2330.TW", "2449.TW"]；空白則讀取最新WHID台股報表。
settings = load_config(CONFIG_PATH)
if TICKERS:
    settings["candidates"]["tickers"] = TICKERS
candidate_frame, source_info = candidates(settings, PROJECT_ROOT)
print(source_info)
display(candidate_frame)


## 分析與輸出

目前依 YAML 處理全部候選；可用 max_stocks 限制檔數。首次需下載及訓練，請等待各股票進度。當日完整資料以台灣20:00後為準。候選名單過舊會要求重新評估。

In [ ]:
result = run(CONFIG_PATH, candidate_frame=candidate_frame, provenance=source_info)
display(result["simple"])
for kind,path in result["paths"].items():
    print(kind, path)


In [ ]:
# 這是單檔時機研究，不是WHID歷史選股或多檔投資組合績效。
display(result["backtest"])
display(result["ai_status"])


## 可選：FinLab研究介接（預設不執行）

需要自行安裝FinLab及登入你有權使用的帳號。這是固定候選股票的歷史研究，不能宣稱WHID歷史選股績效。費稅、部位與成交假設須另外核對；不會下單。

In [ ]:
RUN_FINLAB = False
if RUN_FINLAB:
    import finlab
    from finlab.backtest import sim
    # 如需登入，依FinLab官方流程在本機操作，不要把金鑰寫入Notebook。
    positions = finlab_research_positions(result["signals"])
    # 從每檔研究起日的交集開始，避免不完整期間比較。
    if positions.empty or result["backtest"].empty:
        raise ValueError("沒有可用研究訊號")
    start = pd.to_datetime(result["backtest"]["start"]).max()
    positions = positions.loc[start:]
    finlab_report = sim(positions, trade_at_price="open", position_limit=0.1,
        fee_ratio=settings["backtest"]["fee_rate"], tax_ratio=settings["backtest"]["sell_tax_rate"],
        stop_loss=settings["backtest"]["stop_loss"], take_profit=settings["backtest"]["take_profit"],
        upload=False, name="WHID第二階段固定名單研究")
    display(finlab_report)
